<div style="border: 5px solid black; padding: 20px; border-radius: 6px;">

## **Course:** DSC630 — Predictive Analytics  
## **Name:** Tim Hollis  
## **Assignment:** Exercise 10.2  
## **Date:** May 12, 2026  

</div>

<h2 style="text-align: center;">🎬 Hybrid Movie Recommender System</h2>
<h4 style="text-align: center;">Content-Based + Collaborative Filtering Using the MovieLens Dataset</h4>

## Initial Setup:

In [25]:
# Loading Libraries
import pandas as pd
import numpy as np
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix
warnings.filterwarnings('ignore')

# File Paths
DATA_DIR = 'ml-latest-small/'
MOVIES_PATH = DATA_DIR + 'movies.csv'
RATINGS_PATH = DATA_DIR + 'ratings.csv'
TAGS_PATH = DATA_DIR + 'tags.csv'
LINKS_PATH = DATA_DIR + 'links.csv'

# Model Settings
N_COMPONENTS = 50    # SVD latent factors
MIN_RATINGS = 10    # minimum ratings a movie must have to be recommended
RANDOM_STATE = 42
CONTENT_WEIGHT = 0.40  # genre + tag similarity weight
COLLAB_WEIGHT = 0.60  # collaborative filtering weight
TOP_N = 10    # number of recommendations to return

# Load Data
movies = pd.read_csv(MOVIES_PATH)
ratings = pd.read_csv(RATINGS_PATH)
tags = pd.read_csv(TAGS_PATH)
links = pd.read_csv(LINKS_PATH)

# Preview
print('✅ Libraries imported and data loaded successfully')
print(f'📂 Data directory: {DATA_DIR}')
print(f'⚙️  SVD components: {N_COMPONENTS}')
print(
    f'⚖️  Weights — Content: {CONTENT_WEIGHT} | Collaborative: {COLLAB_WEIGHT}')
print(f'🎬 Recommendations per query: {TOP_N}')
print(f'\n📊 Dataset shapes:')
print(f'   movies:  {movies.shape[0]:,} rows, {movies.shape[1]} columns')
print(f'   ratings: {ratings.shape[0]:,} rows, {ratings.shape[1]} columns')
print(f'   tags:    {tags.shape[0]:,} rows, {tags.shape[1]} columns')
print(f'   links:   {links.shape[0]:,} rows, {links.shape[1]} columns')
print(f'\n🔍 Preview — movies:')
movies.head(3)

✅ Libraries imported and data loaded successfully
📂 Data directory: ml-latest-small/
⚙️  SVD components: 50
⚖️  Weights — Content: 0.4 | Collaborative: 0.6
🎬 Recommendations per query: 10

📊 Dataset shapes:
   movies:  9,742 rows, 3 columns
   ratings: 100,836 rows, 4 columns
   tags:    3,683 rows, 4 columns
   links:   9,742 rows, 3 columns

🔍 Preview — movies:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance


<div style="page-break-after: always;"></div>

## Content-Based Setup: Genre and Tag Similarity

In [26]:
# Combine tags per movie into a single string
tag_data = tags.groupby('movieId')['tag'].apply(
    lambda x: ' '.join(x.astype(str))).reset_index()
tag_data.columns = ['movieId', 'combined_tags']

# Merge tags into movies dataframe
content_df = movies.merge(tag_data, on='movieId', how='left')
content_df['combined_tags'] = content_df['combined_tags'].fillna('')

# Clean genres — replace pipes with spaces so TF-IDF treats each genre as
# a token
content_df['cleaned_genres'] = content_df['genres'].str.replace(
    '|', ' ', regex=False)

# Combine genres and tags into one content string
content_df['content'] = content_df['cleaned_genres'] + \
    ' ' + content_df['combined_tags']

# Build TF-IDF matrix on combined content
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(content_df['content'])

# Compute cosine similarity between all movies
content_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Map movie titles to dataframe index for fast lookup
title_to_idx = pd.Series(content_df.index, index=content_df['title']).to_dict()

print('✅ Content-based similarity matrix built successfully')
print(f'🎭 TF-IDF vocabulary size: {len(tfidf.vocabulary_):,} terms')
print(f'📐 Similarity matrix shape: {content_sim.shape}')
print(
    f'🏷️  Movies with tags: {(content_df["combined_tags"] != "").sum():,} of {len(content_df):,}')
print(f'\n🔍 Sample content string for Toy Story (1995):')
print(
    f'   {content_df[content_df["title"] == "Toy Story (1995)"]["content"].values[0]}')

✅ Content-based similarity matrix built successfully
🎭 TF-IDF vocabulary size: 1,677 terms
📐 Similarity matrix shape: (9742, 9742)
🏷️  Movies with tags: 1,572 of 9,742

🔍 Sample content string for Toy Story (1995):
   Adventure Animation Children Comedy Fantasy pixar pixar fun


## Collaborative Filtering Setup: SVD Matrix Factorization

In [27]:
# Build user-movie ratings matrix
ratings_matrix = ratings.pivot(
    index='userId',
    columns='movieId',
    values='rating').fillna(0)

# Convert to sparse matrix for efficiency
ratings_sparse = csr_matrix(ratings_matrix.values)

# Apply Truncated SVD to decompose the ratings matrix
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
svd_matrix = svd.fit_transform(ratings_sparse)

# Reconstruct the full predicted ratings matrix
ratings_predicted = np.dot(svd_matrix, svd.components_)

# Wrap in a DataFrame with original index and columns
ratings_predicted_df = pd.DataFrame(
    ratings_predicted,
    index=ratings_matrix.index,
    columns=ratings_matrix.columns
)

# Filter to movies meeting minimum rating threshold
movie_rating_counts = ratings.groupby('movieId')['rating'].count()
qualified_movies = movie_rating_counts[movie_rating_counts >=
                                       MIN_RATINGS].index

# Build movie-to-column position lookup for the predicted matrix
movieid_to_col = {mid: i for i, mid in enumerate(ratings_matrix.columns)}

print('✅ Collaborative filtering model built successfully')
print(f'👥 Users in matrix:          {ratings_matrix.shape[0]:,}')
print(f'🎬 Movies in matrix:         {ratings_matrix.shape[1]:,}')
print(f'🧮 SVD latent factors:       {N_COMPONENTS}')
print(f'📊 Variance explained:       {svd.explained_variance_ratio_.sum():.1%}')
print(
    f'✅ Movies meeting min ratings threshold ({MIN_RATINGS}): {len(qualified_movies):,}')

✅ Collaborative filtering model built successfully
👥 Users in matrix:          610
🎬 Movies in matrix:         9,724
🧮 SVD latent factors:       50
📊 Variance explained:       53.9%
✅ Movies meeting min ratings threshold (10): 2,269


## Hybrid Recommender Setup: Content + Collaborative Scoring

In [28]:
# Build movie-to-movie collaborative similarity from SVD components
movie_factors = svd.components_.T
movie_ids_ordered = list(ratings_matrix.columns)

# Map movieId to its SVD row index
movieid_to_svd_idx = {mid: i for i, mid in enumerate(movie_ids_ordered)}

# Compute movie-to-movie cosine similarity from latent factors
collab_sim = cosine_similarity(movie_factors)

collab_sim_df = pd.DataFrame(
    collab_sim,
    index=movie_ids_ordered,
    columns=movie_ids_ordered
)


def get_hybrid_recommendations(movie_title, top_n=TOP_N):
    if movie_title not in title_to_idx:
        close = [t for t in title_to_idx if movie_title.lower() in t.lower()]
        if close:
            print(f'❌ Exact title not found. Did you mean one of these?')
            for c in close[:5]:
                print(f'   → {c}')
        else:
            print(
                f'❌ Movie not found: "{movie_title}". Check spelling and include the year.')
        return None

    # Get content-based similarity scores for this movie
    idx = title_to_idx[movie_title]
    content_scores = {
        i: score for i, score in enumerate(
            content_sim[idx]) if i != idx}

    # Get input movie ID
    input_movie_id = content_df.iloc[idx]['movieId']

    # Get collaborative similarity scores for this movie
    if input_movie_id in collab_sim_df.index:
        collab_scores = collab_sim_df.loc[input_movie_id]
    else:
        collab_scores = pd.Series(dtype=float)

    # Build hybrid score for each candidate movie
    results = []
    for i, c_score in content_scores.items():
        movie_id = content_df.iloc[i]['movieId']

        # Skip input movie and movies below minimum ratings threshold
        if movie_id == input_movie_id:
            continue
        if movie_id not in qualified_movies.values:
            continue

        # Get collaborative score for this specific movie pair
        cf_score = collab_scores[movie_id] if movie_id in collab_scores.index else 0.0

        # Compute weighted hybrid score
        hybrid_score = (CONTENT_WEIGHT * c_score) + (COLLAB_WEIGHT * cf_score)

        # Get IMDB link
        imdb_id = links[links['movieId'] == movie_id]['imdbId'].values
        imdb_link = f'https://www.imdb.com/title/tt{str(imdb_id[0]).zfill(7)}/' if len(
            imdb_id) > 0 else 'N/A'

        results.append({
            'title': content_df.iloc[i]['title'],
            'genres': content_df.iloc[i]['genres'],
            'hybrid_score': round(hybrid_score, 4),
            'content_score': round(c_score, 4),
            'collab_score': round(cf_score, 4),
            'imdb': imdb_link
        })

    # Sort by hybrid score and return top N
    results_df = pd.DataFrame(results).sort_values(
        'hybrid_score',
        ascending=False).head(top_n).reset_index(
        drop=True)
    results_df.index += 1

    print(f'\n🎬 Because you liked "{movie_title}", we recommend:\n')
    for i, row in results_df.iterrows():
        print(f'  {i:>2}. {row["title"]} ({row["genres"]})')
        print(
            f'      Hybrid: {row["hybrid_score"]} | Content: {row["content_score"]} | Collab: {row["collab_score"]}')
        print(f'      🔗 {row["imdb"]}')

    return results_df


print('✅ Hybrid recommender rebuilt with movie-to-movie collaborative similarity')
print('💡 Usage: get_hybrid_recommendations("Toy Story (1995)")')

✅ Hybrid recommender rebuilt with movie-to-movie collaborative similarity
💡 Usage: get_hybrid_recommendations("Toy Story (1995)")


## Example Recommendations

In [29]:
# Example 1 — Comedy/Drama
rec1 = get_hybrid_recommendations('Devil Wears Prada, The (2006)')

print('\n' + '=' * 65)

# Example 2 — Animated/Action
rec2 = get_hybrid_recommendations('Incredibles, The (2004)')

print('\n' + '=' * 65)

# Example 3 — Mystery/Comedy
rec3 = get_hybrid_recommendations('Clue (1985)')

print('\n' + '=' * 65)

# Example 4 — Crime/Drama
rec4 = get_hybrid_recommendations('Boyz N the Hood (1991)')

print('\n' + '=' * 65)

# Example 5 — Fuzzy match demo: auto-resolves to best match and recommends
fuzzy_input = 'Incredibles 2'
close_matches = [t for t in title_to_idx if fuzzy_input.lower() in t.lower()]
if close_matches:
    resolved = close_matches[0]
    print(f'🔍 "{fuzzy_input}" not found exactly — auto-resolving to: "{resolved}"\n')
    rec5 = get_hybrid_recommendations(resolved)


🎬 Because you liked "Devil Wears Prada, The (2006)", we recommend:

   1. Marley & Me (2008) (Comedy|Drama)
      Hybrid: 0.665 | Content: 1.0 | Collab: 0.4417
      🔗 https://www.imdb.com/title/tt0822832/
   2. 17 Again (2009) (Comedy|Drama)
      Hybrid: 0.6517 | Content: 1.0 | Collab: 0.4196
      🔗 https://www.imdb.com/title/tt0974661/
   3. Great Gatsby, The (2013) (Drama)
      Hybrid: 0.631 | Content: 0.6787 | Collab: 0.5992
      🔗 https://www.imdb.com/title/tt1343092/
   4. The Intern (2015) (Comedy)
      Hybrid: 0.6141 | Content: 0.7344 | Collab: 0.5339
      🔗 https://www.imdb.com/title/tt2361509/
   5. Silver Linings Playbook (2012) (Comedy|Drama)
      Hybrid: 0.5987 | Content: 1.0 | Collab: 0.3312
      🔗 https://www.imdb.com/title/tt1045658/
   6. Bridesmaids (2011) (Comedy)
      Hybrid: 0.5982 | Content: 0.7344 | Collab: 0.5074
      🔗 https://www.imdb.com/title/tt1478338/
   7. Julie & Julia (2009) (Comedy|Drama|Romance)
      Hybrid: 0.5873 | Content: 0.6875 | Coll

<div style="page-break-after: always;"></div>

<div style="border: 5px solid black; padding: 20px; border-radius: 6px;">

### How It Works

**Content-Based Filtering** analyzes each movie's genre tags and user-generated text tags using TF-IDF vectorization. Cosine similarity between movie vectors identifies films with similar content profiles.

**Collaborative Filtering** applies Truncated SVD (matrix factorization) to the user-ratings matrix, decomposing it into 50 latent factors that capture hidden patterns in viewing behavior. Movie-to-movie cosine similarity in this latent space identifies films that tend to be rated similarly by the same users.

**Hybrid Scoring** blends both signals using a weighted formula: `Hybrid Score = (0.40 × Content Similarity) + (0.60 × Collaborative Similarity)`

The collaborative weight is intentionally higher because the ratings data in this dataset is rich enough to trust, and behavioral patterns tend to surface cultural and stylistic nuance that genre labels alone cannot capture.

### Key Design Decisions

- **Tags + Genres combined** into a single TF-IDF content string to enrich content similarity beyond genre alone
- **IMDB links** generated for every recommendation using the links.csv file
- **Minimum ratings threshold** of 10 filters out obscure titles with too few ratings to produce reliable collaborative scores
- **Fuzzy title matching** helps users find movies even with incomplete or slightly incorrect input, auto-resolving to the closest match
- **53.9% variance explained** by SVD with 50 components — strong signal capture from the ratings matrix

### Limitations

- Only 16% of movies in this dataset have user tags, meaning content similarity relies heavily on genre for most films
- Genre labels are broad and do not capture cultural, stylistic, or tonal nuance — for example, films sharing the "Comedy" genre may appeal to very different audiences
- The dataset was collected through 2018, so more recent films are not represented

---

### 📜 Attribution
F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. *ACM Transactions on Interactive Intelligent Systems (TiiS)* 5, 4: 19:1–19:19. https://doi.org/10.1145/2827872

### **References:**

Koren, Y., Bell, R., & Volinsky, C. (2009). Matrix factorization techniques for recommender systems. *Computer, 42*(8), 30–37. https://doi.org/10.1109/MC.2009.263

Lops, P., de Gemmis, M., & Semeraro, G. (2011). Content-based recommender systems: State of the art and trends. In F. Ricci, L. Rokach, B. Shapira, & P. B. Kantor (Eds.), *Recommender systems handbook* (pp. 73–105). Springer. https://doi.org/10.1007/978-0-387-85820-3_3

MovieLens. (2024, July 29). GroupLens. https://grouplens.org/datasets/movielens/

</div>

<div style="border: 5px solid green; padding: 20px; border-radius: 6px;">

## Week 10 Reflection 

### Overall Reflection
This project focused on building a functional recommender system using the small MovieLens dataset, with the goal of allowing a user to input a movie title and receive ten recommended movies in return. Completing this assignment provided a deeper understanding of how predictive analytics techniques can be applied to real-world recommendation problems, particularly in domains where user preferences and item characteristics interact in complex ways.

The feedback highlighted that the submission not only met the assignment requirements but exceeded them by implementing a hybrid recommender system that combined content-based filtering and collaborative filtering. This reinforced that the design choices were appropriate and that the explanation of the workflow, limitations, and references was clear and well‑structured. Overall, the project strengthened my understanding of recommender system design and the trade-offs involved in building practical, scalable recommendation pipelines.

---

### Straightforward Aspects
Several components of the assignment aligned naturally with concepts covered earlier in the course, making them relatively smooth to implement:

- **Loading and Preparing the MovieLens Data:** The dataset was well‑structured, and merging ratings, movies, and metadata was a direct process. This made it easy to move quickly into modeling.

- **Content-Based Filtering with TF‑IDF:** Creating a TF‑IDF matrix from movie genres and computing cosine similarity felt intuitive. The method is conceptually simple, computationally efficient, and produces recommendations that are easy to interpret.

- **Collaborative Filtering with Truncated SVD:** Applying SVD to the user–item matrix was straightforward using scikit‑learn. The latent factor representation made it easy to compute movie‑to‑movie similarity scores.

- **Combining Scores into a Hybrid System:** Averaging the content and collaborative similarity scores was a clean and effective way to merge the strengths of both approaches. This hybrid method produced more balanced recommendations than either method alone.

- **Generating and Displaying Recommendations:** Returning movie titles, genres, similarity scores, and IMDb links made the output clear and user‑friendly. Testing the recommender with multiple movie titles confirmed that the system was reusable and not hard‑coded.

---

### More Challenging Aspects
While the core workflow was manageable, several parts of the project required deeper consideration or presented opportunities for improvement:

- **Exact Title Matching:** The recommender function required an exact movie title match. Although I included a fuzzy‑matching helper, the main function did not fully resolve partial or misspelled titles. Improving this would make the system more robust and user‑friendly.

- **Choosing the Number of SVD Components:** I selected 50 latent components for the Truncated SVD model, but the write‑up did not fully justify this choice. A more thorough explanation—or a comparison of performance across different component values—would strengthen the methodological rigor.

- **Unused Intermediate Objects:** The code generated predicted ratings from the SVD decomposition, but the final recommender relied primarily on movie‑to‑movie similarity in latent space. Either removing unused objects or clarifying their purpose would make the workflow cleaner.

- **Balancing Interpretability and Performance:** Content-based filtering is easy to interpret, while collaborative filtering captures deeper patterns but is less transparent. Combining them required careful explanation to ensure the hybrid approach remained understandable.

- **Documenting the Full Process:** The assignment emphasized clearly explaining all steps. Ensuring that each stage—from data loading to hybrid scoring—was fully documented required additional attention to detail.

---

### Summary
This project provided valuable hands‑on experience with designing and implementing a recommender system using both content-based and collaborative filtering techniques. The hybrid approach produced strong recommendations and demonstrated how combining multiple methods can improve predictive performance. The feedback offered helpful guidance for refining the system—particularly around title matching, SVD component justification, and code clarity. Moving forward, these improvements will help strengthen the final version of the recommender and deepen my understanding of predictive analytics in recommendation contexts.

</div>